# 🛒 네이버 쇼핑 전체 대표 인기물품 TOP 10 고객군별 데이터 분석

네이버 데이터랩 쇼핑인사이트 API를 통해 수집된 **네이버 쇼핑 전체 10개 대표 분야 인기 물품의 최근 2일치 고객군(기기, 성별, 연령) 클릭 데이터**를 Pandas DataFrame으로 로드하고 탐색합니다.

---
### 📌 분석 대상 10개 대표 물품
- **패션의류**: 원피스
- **패션잡화**: 크록스
- **화장품/미용**: ahc아이크림
- **디지털/가전**: 냉장고
- **가구/인테리어**: 식탁의자
- **출산/육아**: 물티슈
- **식품**: 추석선물세트
- **스포츠/레저**: 텐트
- **생활/건강**: 마스크
- **디지털/IT**: 노트북

## 1. 라이브러리 임포트 및 환경 설정

In [1]:
import os
import pandas as pd

# 데이터프레임 출력 행/열 수 확대 설정
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.width', 1000)
print('✅ Pandas 버전:', pd.__version__)

✅ Pandas 버전: 3.0.5


## 2. CSV 데이터 파일 로드
`data/` 디렉터리에 저장된 수집 결과 CSV 파일을 자동으로 탐색하여 로드합니다.

In [2]:
csv_filename = 'shopping_insight_전체쇼핑_TOP10물품_고객군분석_2일치.csv'
candidates = [
    csv_filename,
    os.path.join('..', 'data', csv_filename),
    os.path.join('data', csv_filename),
    os.path.join(r'C:\projects\wepscraping-git\data', csv_filename)
]

target_file = None
for path in candidates:
    if os.path.exists(path):
        target_file = path
        break

if target_file:
    df = pd.read_csv(target_file, encoding='utf-8-sig')
    print(f'✅ 데이터 로드 성공: {target_file}')
    print(f'- 총 행 수: {df.shape[0]}행, 컬럼 수: {df.shape[1]}개')
else:
    raise FileNotFoundError(f'❌ CSV 파일을 찾을 수 없습니다: {csv_filename}')

✅ 데이터 로드 성공: ..\data\shopping_insight_전체쇼핑_TOP10물품_고객군분석_2일치.csv
- 총 행 수: 189행, 컬럼 수: 9개


## 3. 데이터프레임 기본 확인 (`head`, `info`, 고유값)

In [3]:
# 상위 10개 행 미리보기
df.head(10)

,순번,카테고리명,카테고리ID,물품검색어,날짜,분석구분,고객군코드,고객군명,클릭비율지수
0,1,패션의류,50000000,원피스,2026-09-05,AGE,20,20대,2.86103
1,1,패션의류,50000000,원피스,2026-09-05,AGE,30,30대,31.74386
2,1,패션의류,50000000,원피스,2026-09-05,AGE,40,40대,57.76566
3,1,패션의류,50000000,원피스,2026-09-05,AGE,50,50대,71.93460
4,1,패션의류,50000000,원피스,2026-09-05,AGE,60,60대 이상,14.85013
5,1,패션의류,50000000,원피스,2026-09-05,DEVICE,mo,모바일,71.50684
6,1,패션의류,50000000,원피스,2026-09-05,DEVICE,pc,PC,6.79452
7,1,패션의류,50000000,원피스,2026-09-05,GENDER,f,여성,71.27718
8,1,패션의류,50000000,원피스,2026-09-05,GENDER,m,남성,2.06358
9,1,패션의류,50000000,원피스,2026-09-06,AGE,10,10대,0.40871


In [ ]:
# 데이터 컬럼 정보 및 결측치 확인
df.info()

In [4]:
# 수집된 10개 대표 물품 목록 확인
items_df = df[['순번', '카테고리명', '물품검색어']].drop_duplicates().sort_values('순번')
items_df.reset_index(drop=True)

,순번,카테고리명,물품검색어
0,1,패션의류,원피스
1,2,패션잡화,크록스
2,3,화장품/미용,ahc아이크림
3,4,디지털/가전,냉장고
4,5,가구/인테리어,식탁의자
5,6,출산/육아,물티슈
6,7,식품,추석선물세트
7,8,스포츠/레저,텐트
8,9,생활/건강,마스크
9,10,디지털/IT,노트북


## 4. 기기별(PC vs 모바일) 이용 비중 분석
각 물품별로 모바일과 PC 중 어떤 기기를 통한 유입이 많은지 비교합니다.

In [5]:
df_device = df[df['분석구분'] == 'DEVICE'].copy()
device_pivot = df_device.pivot_table(
    index=['순번', '카테고리명', '물품검색어'],
    columns='고객군명',
    values='클릭비율지수',
    aggfunc='mean'
).round(2)

device_pivot

,,고객군명,PC,모바일
순번,카테고리명,물품검색어,,
1,패션의류,원피스,10.55,85.75
2,패션잡화,크록스,11.19,89.37
3,화장품/미용,ahc아이크림,3.72,94.83
4,디지털/가전,냉장고,15.21,99.87
5,가구/인테리어,식탁의자,13.25,88.38
6,출산/육아,물티슈,16.15,84.98
7,식품,추석선물세트,26.24,94.13
8,스포츠/레저,텐트,18.06,85.19
9,생활/건강,마스크,15.29,88.13


## 5. 성별(남성 vs 여성) 이용 비중 분석
각 물품별로 여성과 남성 고객층의 선호도를 분석합니다.

In [6]:
df_gender = df[df['분석구분'] == 'GENDER'].copy()
gender_pivot = df_gender.pivot_table(
    index=['순번', '카테고리명', '물품검색어'],
    columns='고객군명',
    values='클릭비율지수',
    aggfunc='mean'
).round(2)

gender_pivot

,,고객군명,남성,여성
순번,카테고리명,물품검색어,,
1,패션의류,원피스,2.68,85.64
2,패션잡화,크록스,87.55,69.52
3,화장품/미용,ahc아이크림,17.27,93.94
4,디지털/가전,냉장고,94.71,99.67
5,가구/인테리어,식탁의자,42.69,91.74
6,출산/육아,물티슈,68.79,87.01
7,식품,추석선물세트,87.22,71.87
8,스포츠/레저,텐트,81.06,44.57
9,생활/건강,마스크,59.70,91.52


## 6. 연령대별(10대~60대) 선호도 분석
10대부터 60대 이상까지 연령대별 클릭 비율 지수를 피벗 테이블로 비교합니다.

In [7]:
df_age = df[df['분석구분'] == 'AGE'].copy()
age_columns = ['10대', '20대', '30대', '40대', '50대', '60대 이상']

age_pivot = df_age.pivot_table(
    index=['순번', '카테고리명', '물품검색어'],
    columns='고객군명',
    values='클릭비율지수',
    aggfunc='mean'
).reindex(columns=age_columns).round(2)

age_pivot

,,고객군명,10대,20대,30대,40대,50대,60대 이상
순번,카테고리명,물품검색어,,,,,,
1,패션의류,원피스,0.41,2.25,39.17,78.88,74.86,20.44
2,패션잡화,크록스,1.47,8.42,35.65,90.78,53.66,9.46
3,화장품/미용,ahc아이크림,1.94,1.70,19.42,84.47,95.87,65.53
4,디지털/가전,냉장고,NaN,11.30,37.50,98.70,75.33,37.07
5,가구/인테리어,식탁의자,1.17,2.64,43.05,87.77,68.20,33.46
6,출산/육아,물티슈,NaN,4.27,50.41,86.38,63.41,19.72
7,식품,추석선물세트,NaN,10.15,43.19,91.71,65.97,18.81
8,스포츠/레저,텐트,NaN,4.91,35.51,81.78,44.63,22.20
9,생활/건강,마스크,1.77,7.96,16.81,64.16,75.66,54.42


## 7. 10개 대표 물품별 클릭비율 통계 요약 (`describe`)
수집된 클릭비율지수의 수치적 분포를 확인합니다.

In [8]:
df.groupby(['카테고리명', '물품검색어'])['클릭비율지수'].describe().round(2)

,,count,mean,std,min,25%,50%,75%,max
카테고리명,물품검색어,,,,,,,,
가구/인테리어,식탁의자,19.0,49.66,35.24,1.17,17.92,54.24,76.15,100.0
디지털/IT,노트북,20.0,57.09,32.43,3.48,30.60,51.02,82.77,100.0
디지털/가전,냉장고,18.0,63.26,36.74,10.00,36.58,75.33,99.35,100.0
생활/건강,마스크,20.0,47.54,34.60,1.77,14.73,53.05,69.52,100.0
스포츠/레저,텐트,18.0,46.43,31.52,2.34,25.88,40.42,63.19,100.0
식품,추석선물세트,18.0,56.59,32.51,7.43,25.84,65.97,81.17,100.0
출산/육아,물티슈,18.0,53.46,33.21,3.66,19.03,55.90,76.55,100.0
패션의류,원피스,19.0,42.15,37.53,0.41,5.04,31.74,71.72,100.0
패션잡화,크록스,20.0,45.71,36.63,1.34,9.71,45.67,76.00,100.0
